In [1]:
import numpy as np
import rasterio
from pathlib import Path
import geopandas as gpd
from rasterio.features import rasterize
import numpy as np
from sklearn.metrics import jaccard_score
import json
import pandas as pd

In [ ]:
def remove_pixels(reference_tiff_path, geojson_path, sample_points_gdf, all_touched=True):
    """Removes pixels from the reference TIFF that are inside the geometries defined 
    in the county GeoJSON file, but not at the sample points."""

    # Load the GeoJSON file using geopandas
    gdf = gpd.read_file(geojson_path)

    # Open the reference TIFF and rasterize the geometries to create a mask
    with rasterio.open(reference_tiff_path) as src:
        if gdf.crs is not None and src.crs is not None and gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)

        # Filter out invalid geometries
        geometries = [geom for geom in gdf.geometry if geom is not None and not geom.is_empty]
        if len(geometries) == 0:
            raise ValueError("No valid geometries found in geojson_path.")

        # Rasterize the geometries to create a mask of pixels inside the geometries
        inside_mask = rasterize(
            shapes=[(geom, 1) for geom in geometries],
            out_shape=(src.height, src.width),
            transform=src.transform,
            fill=0,
            all_touched=all_touched,
            dtype="uint8",
        ).astype(bool)

        # Rasterize the sample points to create a mask of pixels at the sample points
        geometries_points = sample_points_gdf.geometry
        points_mask = rasterize(
            shapes=[(geom, 1) for geom in geometries_points],
            out_shape=(src.height, src.width),
            transform=src.transform,
            fill=0,
            all_touched=all_touched,
            dtype="uint8",
        ).astype(bool)

        # Combine the masks to identify pixels that are inside the geometries but not at the sample points
        combined_mask = inside_mask & ~points_mask
        inside_pixels_combined = src.read(1)[combined_mask]


    return inside_pixels_combined

In [ ]:
def calculate_pixel_iou_save(raster_path, true_labels, save_path, geojson_path, sample_points_gdf):
    '''Calculates the pixel-wise IoU between the true labels and the predicted labels 
    obtained from the remove_pixels function, and saves the metrics to a JSON file.'''

    # Flatten the true labels to a 1D array
    y_true = true_labels.reshape(-1)

    # Get the predicted labels by removing pixels based on the county GeoJSON and sample points
    y_pred = remove_pixels(raster_path, geojson_path, sample_points_gdf, all_touched=True)

    # Calculate the IoU for each class and the mean IoU
    classes = np.union1d(y_true, y_pred)
    class_iou = jaccard_score(y_true, y_pred, labels=classes, average=None, zero_division=0)
    class_iou = dict(zip(classes.tolist(), class_iou.tolist()))
    miou = float(np.mean(list(class_iou.values())))

    # Calculate the foreground IoU (considering all non-zero classes as foreground)
    fg_iou = jaccard_score(
        (y_true != 0).astype(np.uint8),
        (y_pred != 0).astype(np.uint8),
        zero_division=0,
    )

    # Print the metrics
    print("Class IoU:", class_iou)
    print("mIoU:", miou)
    print("Foreground IoU:", fg_iou)

    # Save the metrics to a JSON file
    metrics = {
        "class_iou": class_iou,
        "miou": miou,
        "foreground_iou": fg_iou,
    }

    # Ensure the save path exists and load existing metrics if the file already exists
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    # If the file already exists, load the existing metrics and update them with the new metrics
    existing_metrics = {}
    with open(save_path, "r") as f:
                existing_metrics = json.load(f)
    existing_metrics.update(metrics)
    
    # Save the updated metrics back to the JSON file
    with open(save_path, "w") as f:
        json.dump(existing_metrics, f, indent=2)

In [ ]:
# define paths and constants
WORK_DIR = Path("/Users/muhammadabdul/Desktop/penn/earth_genome_internship/climate_trace")
raster_path = WORK_DIR / "sub_facility_discrimination_KS_sp" / "inference" / "20260309_150216" / "merged_processed.tif"
save_path = WORK_DIR / "sub_facility_discrimination_KS_sp" / "model" / "20260309_150216"/ "model_metrics.json"
county_poly_path = WORK_DIR / "data" / "KS_barton_lot_pond" / "county_poly.geojson"
lot_sample_points = WORK_DIR / "data" / "KS_barton_lot_pond" / "lot_sample_points_sp.geojson"
pond_sample_points = WORK_DIR / "data" / "KS_barton_lot_pond" / "pond_sample_points_sp.geojson"
other_sample_points = WORK_DIR / "data" / "KS_barton_lot_pond" / "other_sample_points_sp.geojson"
geojson_path = WORK_DIR / "data" / "KS_barton_lot_pond" / "lot_pond_combined.geojson"  
CLASS_FIELD = "class" 

In [5]:
# combine sample points into one GeoDataFrame
lot_gdf = gpd.read_file(lot_sample_points)
pond_gdf = gpd.read_file(pond_sample_points)
other_gdf = gpd.read_file(other_sample_points)
sample_points_gdf = gpd.GeoDataFrame(pd.concat([lot_gdf, pond_gdf, other_gdf], ignore_index=True))

In [6]:
# Create an in-memory GeoTIFF with the same metadata/resolution as raster_path, filled with zeros
with rasterio.open(raster_path) as src:
    profile = src.profile.copy()
    zero_data = np.zeros((src.count, src.height, src.width), dtype=src.dtypes[0])

# Write the zero-filled data to an in-memory GeoTIFF
mem_tif = rasterio.io.MemoryFile()
tmp_tif = mem_tif.open(**profile)
tmp_tif.write(zero_data)

In [ ]:
# define class mapping for rasterization
class_to_code = {"lot": 2, "pond": 1, "other": 0}

# Read vector labels
gdf = gpd.read_file(geojson_path)

# Normalize class values and map to numeric codes (unknown classes -> "other" -> 0)
gdf[CLASS_FIELD] = gdf[CLASS_FIELD].astype(str).str.lower()
gdf["code"] = gdf[CLASS_FIELD].map(class_to_code).fillna(class_to_code["other"]).astype("uint8")

# Build (geometry, value) pairs for rasterization
shapes = [(geom, code) for geom, code in zip(gdf.geometry, gdf["code"]) if geom is not None and not geom.is_empty]

# Rasterize so any intersecting pixel is assigned a class (all_touched=True)
coded = rasterize(
    shapes=shapes,
    out_shape=(profile["height"], profile["width"]),
    transform=profile["transform"],
    fill=class_to_code["other"],
    all_touched=True,
    dtype="uint8"
)

# Write into your existing in-memory GeoTIFF
tmp_tif.write(coded, 1)

# write to disk for inspection
path_to_rasterized_labels = WORK_DIR / "data" / "KS_barton_lot_pond" /  "rasterized_labels.tif"
with rasterio.open(path_to_rasterized_labels, "w", **profile) as dst:
    dst.write(coded, 1)

# remove pixels not in county or intersecting with sample points
true_labels = remove_pixels(path_to_rasterized_labels, county_poly_path, sample_points_gdf,all_touched=True )

In [8]:
calculate_pixel_iou_save(raster_path, true_labels, save_path, county_poly_path, sample_points_gdf)

Class IoU: {0.0: 0.9992202537250892, 1.0: 0.49835895664190705, 2.0: 0.6878511468041509}
mIoU: 0.7284767857237157
Foreground IoU: 0.6618452894217128
